In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import ydata_profiling
from datetime import datetime, timedelta

In [3]:
df = pd.read_csv('datasets/data.csv')

df.head(3)

,No,Water Meter ID,Reading ID,Reading Value,Reading Date,Previous Reading Value,Previous Reading Date,Reading Frequency,Reader ID,Type of Contract,Reading Validity,Certification on the ERP,Final Billing,Reason for Reading
0,1,IT-WM-001,READ-001,145.50,15/01/2024,120.30,15/12/2023,Monthly,READER-01,Residential,Valid,Yes,125.40,Routine
1,2,IT-WM-002,READ-002,89.75,16/01/2024,75.20,16/12/2023,Monthly,READER-02,Commercial,Valid,Yes,89.75,Routine
2,3,IT-WM-003,READ-003,NaN,17/01/2024,210.45,17/12/2023,Monthly,READER-03,Industrial,Invalid,No,0.00,Missing


Transformacion de variables

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 501 entries, 0 to 500
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   No                        501 non-null    int64  
 1   Water Meter ID            501 non-null    object 
 2   Reading ID                501 non-null    object 
 3   Reading Value             494 non-null    float64
 4   Reading Date              501 non-null    object 
 5   Previous Reading Value    494 non-null    float64
 6   Previous Reading Date     496 non-null    object 
 7   Reading Frequency         501 non-null    object 
 8   Reader ID                 499 non-null    object 
 9   Type of Contract          500 non-null    object 
 10  Reading Validity          501 non-null    object 
 11  Certification on the ERP  500 non-null    object 
 12  Final Billing             484 non-null    float64
 13  Reason for Reading        478 non-null    object 
dtypes: float64

In [5]:
# dates transform
times = ['Reading Date', 'Previous Reading Date']

for i in times:
    df[i] = pd.to_datetime(df[i], errors='coerce')
    df[i] = df[i].dt.strftime('%Y-%m-%d')
    print(df[i].head(3))
    print('\n')

0    2024-01-15
1    2024-01-16
2    2024-01-17
Name: Reading Date, dtype: object


0    2023-12-15
1    2023-12-16
2    2023-12-17
Name: Previous Reading Date, dtype: object




/tmp/ipykernel_19141/3314024531.py:5: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[i] = pd.to_datetime(df[i], errors='coerce')
/tmp/ipykernel_19141/3314024531.py:5: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[i] = pd.to_datetime(df[i], errors='coerce')


In [6]:
# bool transform
# iba a agregar final billing pero el dataset no cumple con las condiciones del diccionario
# asi que lo voy a dejar como esta
bools = ['Certification on the ERP']
for i in bools:
    df[i] = df[i].str.lower()
    df[i] = df[i].replace({'true': True, 'false': False, '0': False, '1': True})
    df[i] = df[i].astype(bool)
    print(df[i].head(3))
    print('\n')

# vality
# Replace values in 'Reading Validity' column using the replace() method
df['Reading Validity'] = df['Reading Validity'].replace(
    {'Valid': True, 'Invalid': False, 'Válido': True, 'Inválido': False, 'Valido': True, 'Sospetto': False}
)
df['Reading Validity'] = df['Reading Validity'].astype(bool)
print(df['Reading Validity'].head(3))

0    True
1    True
2    True
Name: Certification on the ERP, dtype: bool


0     True
1     True
2    False
Name: Reading Validity, dtype: bool


Analisis de Variables nulas

In [16]:
null_count = {}

for col in df.columns:
    count = df[col].isnull().sum()
    if count > 0:
        null_count[col]=count
for col, count in null_count.items():
    print(col,":",count)


Reading Value : 7
Previous Reading Value : 7
Previous Reading Date : 13
Reader ID : 2
Type of Contract : 1
Final Billing : 17
Reason for Reading : 23


In [20]:
df.head(1)

,No,Water Meter ID,Reading ID,Reading Value,Reading Date,Previous Reading Value,Previous Reading Date,Reading Frequency,Reader ID,Type of Contract,Reading Validity,Certification on the ERP,Final Billing,Reason for Reading
0,1,IT-WM-001,READ-001,145.5,2024-01-15,120.3,2023-12-15,Monthly,READER-01,Residential,True,True,125.4,Routine


Pero antes si queremos estandarizar usando mediana y demas debemos establecer umbrales y demas

In [ ]:
# reglas para analizar umbrales
validation_rules = {
    # numeric
    "Reading_value": {
        "type": "numeric", 
        "min": 0, 
        "max": 50000, 
        "allow_null": False, 
        "fill": "median"
    },
    "Previous Reading Value": {
        "type": "numeric",
        "min": 0,
        "max": 50000,
        "allow_null": True,
        "fill":  "median"
    },
    "Final Billing":{
        "type": "numeric",
        "min": 0.0,
        "max": 500.0,
        "allow_null": False,
        "fill":  "median"
    },
    # categorical
    "Type of Contract": {
        "type": "categorical",
        "allowed_values": ["residential", "comercial", "industrial"], 
        "allow_null": False,
        "fill_strategy": "mode",
        "mapping": {
            # agregar los de diferentes idiomas
            "residencial": "residential",
            "residenziale": "residential",
            "commerciale": "comercial",
            "industriale": "industrial",
        }
    },
    "Reading Validity":{
        "type": "categorical",
        "allowed_values": ["valid", "invalid"], 
        "allow_null": False,
        "fill_strategy": "mode",
        "mapping": {
            # agregar los de diferentes idiomas
            "suspicious": "invalid",
            "valido": "valid",
            "invalido": "invalid",
            "sospechoso": "invalid",
            "non valido": "invalid",
            "sospetto": "invalid",

        }
    },
    "Certification on the ERP":{
        "type": "categorical",
        "allowed_values": ["yes", "no"], 
        "allow_null": False,
        "fill_strategy": "mode",
        "mapping": {
            # agregar los de diferentes idiomas
            "si": "yes",
            "y": "yes",
            "1": "yes",
            "0": "no",
        }
    },
    "Reading Frequency": {
        "type": "categorical", 
        "allowed_values": ["monthly", "bimonthly"],
        "allow_null": True,
        "fill": "mode",
        "mapping": {
            "mensual": "monthly",
            "bimestral": "bimonthly",
            "mensile": "monthly",
            "bimestrale": "bimonthly",
        }
    },
    "Reason for Reading":{
        "type": "categorical",
        "allowed_values": [], 
        "allow_null": True,
        "fill_strategy": "mode",
    },
}